# Phase 3 — Feature Engineering & Train/Test Split: Validation

**Status:** ✅ Completed (pipeline v1.0)

This notebook is a **read-only validation layer** over `ml_service/src/feature_engineer.py`.
It loads `artifacts/splits/split_summary_v1.json` and inspects split artifacts to verify:
- 3-pass streaming execution and RAM constraint compliance (< 500 MB limit)
- 80/20 stratified split row accounting across all 15 categories
- Exact stratification alignment between train and test distributions
- Scaling discipline (train-only fit, float32 encoding)
- Artifact integrity in `artifacts/splits/`

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import numpy as np

root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from config.paths import Paths

with open(Paths.SPLITS_DIR / 'split_summary_v1.json', 'r', encoding='utf-8') as f:
    summary = json.load(f)

print(f"Phase:            {summary['phase']}")
print(f"Pipeline version: {summary['pipeline_version']}")
print(f"Timestamp (UTC):  {summary['timestamp_utc']}")
print(f"Elapsed:          {summary['elapsed_seconds']} s")
print(f"Peak RSS:         {summary['memory']['peak_rss_mb']} MB (budget: {summary['memory']['memory_limit_mb']} MB, within_limit={summary['memory']['within_limit']})")

## 1. Split Accounting & Row Breakdown

In [ ]:
rc = summary['row_counts']
le_meta = summary['label_encoding']

split_df = pd.DataFrame([
    {'Split': 'Train (80%)', 'Rows': rc['train'], 'Fraction': f"{rc['train']/rc['total']*100:.1f}%"},
    {'Split': 'Test (20%)', 'Rows': rc['test'], 'Fraction': f"{rc['test']/rc['total']*100:.1f}%"},
    {'Split': 'Total Clean Flows', 'Rows': rc['total'], 'Fraction': '100.0%'}
])
split_df.style.format({'Rows': '{:,}'})

## 2. 15-Class Stratification Audit (Train vs. Test)

In [ ]:
train_dist = summary['class_distribution']['train']
test_dist  = summary['class_distribution']['test']
classes = summary['label_encoding']['category_classes']

dist_rows = []
for cls in classes:
    n_tr = train_dist[cls]
    n_te = test_dist[cls]
    pct_tr = (n_tr / rc['train']) * 100
    pct_te = (n_te / rc['test']) * 100
    dist_rows.append({
        'Category': cls,
        'Train Count': n_tr,
        'Train %': f"{pct_tr:.3f}%",
        'Test Count': n_te,
        'Test %': f"{pct_te:.3f}%",
        'Delta %': f"{abs(pct_tr - pct_te):.4f}%"
    })

pd.DataFrame(dist_rows).style.format({'Train Count': '{:,}', 'Test Count': '{:,}'})

## 3. Generated Split Artifacts Inventory

In [ ]:
artifacts = summary['artifacts']
art_rows = [{'Artifact Key': k, 'Path': v} for k, v in artifacts.items()]
pd.DataFrame(art_rows)